# 🧱 Lab: Multi-Step Reasoning Engine

**Module 3: Prompt Engineering** | **Duration: ~1.5 hours** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Implement** Chain-of-Thought prompting for complex reasoning tasks
2. **Apply** Self-Consistency sampling for more robust answers
3. **Build** prompt chains that decompose multi-step problems
4. **Analyze** when each reasoning strategy outperforms direct prompting

## Concepts Covered

| Concept | From |
|---------|------|
| Chain-of-Thought | mini-cot |
| Self-Consistency | This lab |
| Prompt Chaining | mini-chaining |

## Prerequisites

- **mini-cot**: Chain-of-Thought prompting fundamentals
- **mini-chaining**: Prompt chaining basics
- OpenAI API key configured in `.env`

In [2]:
from openai import OpenAI
from dotenv import load_dotenv
from collections import Counter
import re
from IPython.display import Markdown, display

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

load_dotenv()
client = OpenAI()

# Test problems
# Answer is 11
MATH_PROBLEM = '''Roger has 5 tennis balls. He buys 2 more cans of tennis balls.
Each can has 3 tennis balls. How many tennis balls does he have now?'''

# Harder problem that models often get wrong (correct answer: 5 cents)
HARD_MATH_PROBLEM = '''A bat and a ball cost $1.10 together. 
The bat costs $1.00 more than the ball. 
How much does the ball cost in cents?'''

# Another tricky problem (correct answer: 60 students)
TRICKY_PROBLEM = '''In a school, 1/2 of the students study French, 1/3 study Spanish, 
and 1/6 study both languages. If 20 students study neither language, 
how many students are in the school?'''

LOGIC_PROBLEM = '''If all roses are flowers and some flowers fade quickly,
can we conclude that some roses fade quickly?'''

print('Problems loaded!')
print(f'Easy problem: Tennis balls')
print(f'Hard problem: Bat and ball (tricky!)')  
print(f'Tricky problem: Language students')

Problems loaded!
Easy problem: Tennis balls
Hard problem: Bat and ball (tricky!)
Tricky problem: Language students


## 2. Chain-of-Thought Prompting

CoT encourages the model to break down problems into steps.

In [3]:
def solve_with_cot(problem):
    '''Solve problem using Chain-of-Thought prompting.'''
    prompt = f'''Solve this problem step by step.

Problem: {problem}

Let me work through this step by step:'''
    
    response = client.chat.completions.create(
        model='gpt-3.5-turbo',
        messages=[{'role': 'user', 'content': prompt}])
    return response.choices[0].message.content

def solve_without_cot(problem):
    '''Solve problem directly without Chain-of-Thought prompting.'''
    prompt = f'''Answer this question. Just report the final answer.:

{problem}'''
    
    response = client.chat.completions.create(
        model='gpt-3.5-turbo',
        messages=[{'role': 'user', 'content': prompt}])
    return response.choices[0].message.content

md("\n---\n\n> **Notice:** Without CoT, the model may give a correct answer but doesn't show its reasoning. For more complex problems, this approach is more likely to make errors.")


---

> **Notice:** Without CoT, the model may give a correct answer but doesn't show its reasoning. For more complex problems, this approach is more likely to make errors.

In [4]:
md(f"\n**Problem:** {MATH_PROBLEM}")
md("> **Correct answer: 11 tennis balls**")

# Solve math problem with CoT
md("### 🧮 Math Problem with CoT")
md(solve_with_cot(MATH_PROBLEM))

# Solve math problem without CoT
md("### 🧮 Math Problem without CoT")
md(solve_without_cot(MATH_PROBLEM))


**Problem:** Roger has 5 tennis balls. He buys 2 more cans of tennis balls.
Each can has 3 tennis balls. How many tennis balls does he have now?

> **Correct answer: 11 tennis balls**

### 🧮 Math Problem with CoT



Step 1: Roger starts with 5 tennis balls.
Step 2: He buys 2 cans of tennis balls, each containing 3 tennis balls.
Step 3: Each can has 3 tennis balls, so 2 cans would have 2 * 3 = 6 tennis balls.
Step 4: Adding the initial 5 tennis balls to the 6 from the cans, Roger now has a total of 5 + 6 = 11 tennis balls.

Therefore, Roger now has 11 tennis balls.

### 🧮 Math Problem without CoT

11 tennis balls

In [5]:
md(f"\n**Problem:** {TRICKY_PROBLEM}")
md("> **Correct answer: 60 students**")

# Solve tricky problem with CoT
md("### ✅ Tricky Problem with CoT")
md(solve_with_cot(TRICKY_PROBLEM))

# Solve tricky problem without CoT
md("### ❌ Tricky Problem WITHOUT CoT (Direct)")
md(solve_without_cot(TRICKY_PROBLEM))


**Problem:** In a school, 1/2 of the students study French, 1/3 study Spanish, 
and 1/6 study both languages. If 20 students study neither language, 
how many students are in the school?

> **Correct answer: 60 students**

### ✅ Tricky Problem with CoT

1. Let the total number of students in the school be represented by "x".
2. Given that 1/2 of the students study French, the number of students studying French is (1/2)x.
3. Given that 1/3 of the students study Spanish, the number of students studying Spanish is (1/3)x.
4. Given that 1/6 of the students study both languages, the number of students studying both French and Spanish is (1/6)x.
5. We can represent the total number of students studying either French or Spanish as follows:
   (1/2)x + (1/3)x - (1/6)x = (5/6)x
6. We are also given that 20 students study neither language. Therefore, the number of students studying at least one language is x - 20.
7. Setting the number of students studying at least one language equal to the total number of students studying either French or Spanish, we have:
   x - 20 = (5/6)x
8. Solving for x, we get:
   6(x - 20) = 5x
   6x - 120 = 5x
   x = 120
9. Therefore, there are 120 students in the school.

### ❌ Tricky Problem WITHOUT CoT (Direct)

60 students

In [6]:
# Now let's try the HARD problem - this is where CoT really matters!
md("---\n## 🎯 Hard Problem: Bat and Ball (Classic Cognitive Trap)")
md("> **Correct answer: 5 cents** (most people intuitively say 10 cents)")

md("\n### ❌ WITHOUT Chain-of-Thought:")
md(solve_without_cot(HARD_MATH_PROBLEM))

md("\n### ✅ WITH Chain-of-Thought:")
md(solve_with_cot(HARD_MATH_PROBLEM))

---
## 🎯 Hard Problem: Bat and Ball (Classic Cognitive Trap)

> **Correct answer: 5 cents** (most people intuitively say 10 cents)


### ❌ WITHOUT Chain-of-Thought:

The ball costs 5 cents.


### ✅ WITH Chain-of-Thought:



Let's call the cost of the ball "x" in cents.

According to the problem, the bat costs $1 more than the ball. So, the cost of the bat would be x + 100 cents.

Together, the cost of the bat and the ball is $1.10. So we can set up the following equation:

x + (x + 100) = 110

Combining like terms, we get:

2x + 100 = 110

Subtract 100 from both sides:

2x = 10

Divide by 2:

x = 5

Therefore, the ball costs 5 cents.

In [7]:
# Zero-shot CoT: Just add 'Let's think step by step'
def zero_shot_cot(problem):
    prompt = f"{problem}\n\nLet's think step by step."
    response = client.chat.completions.create(
        model='gpt-3.5-turbo',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.2
    )
    return response.choices[0].message.content

md("### 🧠 Zero-Shot CoT on Logic Problem")
md(zero_shot_cot(LOGIC_PROBLEM))

### 🧠 Zero-Shot CoT on Logic Problem

1. All roses are flowers: This means that every rose is a type of flower.

2. Some flowers fade quickly: This statement does not specify which flowers fade quickly, only that there are some flowers that do.

Based on the given information, we cannot definitively conclude that some roses fade quickly. Just because all roses are flowers does not necessarily mean that they share the same characteristics as all flowers. It is possible that roses have different properties or characteristics that affect their longevity compared to other flowers.

## 3. Self-Consistency Sampling

Generate multiple solutions and take the majority answer.

In [8]:
def extract_number(text):
    '''Extract the final numerical answer.'''
    numbers = re.findall(r'\b(\d+)\b', text)
    return int(numbers[-1]) if numbers else None

def solve_with_cot_temp(problem, temperature=0.2):
    '''Solve problem using CoT with configurable temperature.'''
    prompt = f'''Solve this problem step by step.

Problem: {problem}

Let me work through this step by step:'''
    
    response = client.chat.completions.create(
        model='gpt-3.5-turbo',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=temperature
    )
    return response.choices[0].message.content

def self_consistent_solve(problem, n_samples=5, temperature=0.2):
    '''Solve using self-consistency: generate multiple answers and vote.
    
    Uses higher temperature to get diverse reasoning paths.
    '''
    answers = []
    
    for i in range(n_samples):
        solution = solve_with_cot_temp(problem, temperature=temperature)
        answer = extract_number(solution)
        answers.append(answer)
        print(f'Sample {i+1}: Answer = {answer}')
    
    # Majority vote
    counter = Counter([a for a in answers if a is not None])
    if counter:
        final_answer = counter.most_common(1)[0][0]
        confidence = counter[final_answer] / n_samples
        return final_answer, confidence
    return None, 0

md("### 🎲 Self-Consistency (5 samples)")
answer, conf = self_consistent_solve(MATH_PROBLEM, n_samples=5, temperature=1)
md(f"\n---\n\n**Final Answer:** `{answer}` (confidence: **{conf:.0%}**)")

### 🎲 Self-Consistency (5 samples)

Sample 1: Answer = 11
Sample 2: Answer = 11
Sample 3: Answer = 11
Sample 4: Answer = 11
Sample 5: Answer = 11



---

**Final Answer:** `11` (confidence: **100%**)

In [10]:
# Now try Self-Consistency on a HARDER problem where models may disagree
md("### 🎲 Self-Consistency on TRICKY Problem (Language Students)")
md("> **Correct answer: 60 students**")
md(f"\n**Problem:** {TRICKY_PROBLEM}")

answer, conf = self_consistent_solve(TRICKY_PROBLEM, n_samples=5, temperature=1)
md(f"\n---\n\n**Final Answer:** `{answer}` (confidence: **{conf:.0%}**)")

if conf < 1.0:
    md("\n> ⚠️ **Notice:** The model gave different answers across samples! This shows why self-consistency is valuable - it reveals uncertainty.")

### 🎲 Self-Consistency on TRICKY Problem (Language Students)

> **Correct answer: 60 students**


**Problem:** In a school, 1/2 of the students study French, 1/3 study Spanish, 
and 1/6 study both languages. If 20 students study neither language, 
how many students are in the school?

Sample 1: Answer = 60
Sample 2: Answer = 120
Sample 3: Answer = 60
Sample 4: Answer = 6
Sample 5: Answer = 60



---

**Final Answer:** `60` (confidence: **60%**)


> ⚠️ **Notice:** The model gave different answers across samples! This shows why self-consistency is valuable - it reveals uncertainty.

## 4. Prompt Chaining

Break complex tasks into a sequence of simpler prompts.

In [11]:
def prompt_chain(task):
    '''Execute a multi-step task using prompt chaining.'''
    
    # Step 1: Understand and break down the problem
    step1 = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': f'Identify the key components of this problem: {task}'}],
        temperature=0
    ).choices[0].message.content
    md("#### Step 1 - Problem Analysis")
    md(step1)
    
    # Step 2: Plan solution approach
    step2 = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': f'Based on this analysis: {step1}\n\nOutline the solution steps:'}],
        temperature=0
    ).choices[0].message.content
    md("---\n#### Step 2 - Solution Plan")
    md(step2)
    
    # Step 3: Execute and get final answer
    step3 = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': f'Follow this plan: {step2}\n\nExecute and give the final answer:'}],
        temperature=0
    ).choices[0].message.content
    md("---\n#### Step 3 - Final Answer")
    md(step3)
    return step3

COMPLEX_PROBLEM = '''A store sells apples for $2 each and oranges for $3 each.
John spends $24 total and buys 10 fruits. How many of each did he buy?'''

md("### 🔗 Prompt Chaining Example")
prompt_chain(COMPLEX_PROBLEM)

### 🔗 Prompt Chaining Example

#### Step 1 - Problem Analysis

To solve the problem, we need to identify the key components:

1. **Variables**:
   - Let \( x \) be the number of apples John buys.
   - Let \( y \) be the number of oranges John buys.

2. **Cost of Fruits**:
   - Apples cost $2 each.
   - Oranges cost $3 each.

3. **Total Cost**:
   - John spends a total of $24.

4. **Total Number of Fruits**:
   - John buys a total of 10 fruits.

5. **Equations**:
   - From the total number of fruits: 
     \[
     x + y = 10
     \]
   - From the total cost:
     \[
     2x + 3y = 24
     \]

These components can be used to set up a system of equations to solve for \( x \) and \( y \).

---
#### Step 2 - Solution Plan

To solve the system of equations for the number of apples \( x \) and the number of oranges \( y \) that John buys, we can follow these steps:

1. **Set Up the Equations**:
   - We have the two equations:
     \[
     x + y = 10 \quad \text{(1)}
     \]
     \[
     2x + 3y = 24 \quad \text{(2)}
     \]

2. **Solve for One Variable**:
   - From equation (1), we can express \( y \) in terms of \( x \):
     \[
     y = 10 - x \quad \text{(3)}
     \]

3. **Substitute into the Second Equation**:
   - Substitute equation (3) into equation (2):
     \[
     2x + 3(10 - x) = 24
     \]

4. **Simplify the Equation**:
   - Distribute the 3:
     \[
     2x + 30 - 3x = 24
     \]
   - Combine like terms:
     \[
     -x + 30 = 24
     \]

5. **Isolate the Variable**:
   - Subtract 30 from both sides:
     \[
     -x = 24 - 30
     \]
     \[
     -x = -6
     \]
   - Multiply by -1:
     \[
     x = 6
     \]

6. **Find the Other Variable**:
   - Substitute \( x = 6 \) back into equation (3) to find \( y \):
     \[
     y = 10 - 6 = 4
     \]

7. **Conclusion**:
   - John buys \( x = 6 \) apples and \( y = 4 \) oranges.

8. **Verification**:
   - Check the total number of fruits:
     \[
     x + y = 6 + 4 = 10 \quad \text{(correct)}
     \]
   - Check the total cost:
     \[
     2x + 3y = 2(6) + 3(4) = 12 + 12 = 24 \quad \text{(correct)}
     \]

Thus, the solution is verified, and John buys 6 apples and 4 oranges.

---
#### Step 3 - Final Answer

John buys 6 apples and 4 oranges.

'John buys 6 apples and 4 oranges.'

## 🎯 Summary

### What You Built

You built a multi-step reasoning engine that applies three advanced prompting strategies — Chain-of-Thought, Self-Consistency sampling, and Prompt Chaining — to solve math, logic, and multi-step problems that direct prompting often gets wrong.

### Key Takeaways

1. **Chain-of-Thought** — adding "step by step" reasoning dramatically improves accuracy on math and logic problems
2. **Zero-shot CoT** — simply appending "Let's think step by step" activates reasoning without hand-crafted examples
3. **Self-Consistency** — generating multiple solutions and majority-voting reveals model uncertainty and improves robustness
4. **Prompt Chaining** — breaking complex tasks into sequential prompts lets each step build on the previous one's output